In [ ]:
import json
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
import joblib
from sklearn.metrics import f1_score, classification_report
import os
from google.colab import drive
from sklearn.base import clone
import time

In [ ]:
drive.mount('/content/drive')

output_dir = "/content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_SVC_final"
os.makedirs(output_dir, exist_ok=True)

split_path = "/content/drive/MyDrive/thesis_results/SCOTBESS_splits/SCOTBESS_FULL_ANNOTATED_TERRA_LOW_SPLIT.csv"
mlb_path = "/content/drive/MyDrive/thesis_results/SCOTBESS_splits/scotbess_mlb.joblib"


Mounted at /content/drive


# Data inspection


In [ ]:
scotbess_df = pd.read_csv(split_path)
scotbess_df["label_list"] = scotbess_df["labels"].apply(json.loads)

scotbess_df.head()


,document_id,project,filename,source,final_masked_text,labels,split,label_list
0,0,Aberdeen_City_210665_DPP,210665_DPP-Objects-Mr_Marc_Evans_FULL_-2025345...,comment_structure,The ground on which the proposed energy site i...,"[""Site Selection""]",test,[Site Selection]
1,1,Aberdeen_City_210665_DPP,extracted_210665_DPP-Objects_-_Ken_Cumming__Ve...,no_structure,We have serious reservations that there has no...,"[""Fire and Explosion Risk"", ""Consultation, Tra...",train,"[Fire and Explosion Risk, Consultation, Transp..."
2,2,Aberdeen_City_220026_DPP,extracted_220026_DPP-Objects_-_Brodies_LLP__on...,no_structure,[ORGANIZATION] opposes the application on the ...,"[""Fire and Explosion Risk"", ""Emergency Plannin...",train,"[Fire and Explosion Risk, Emergency Planning a..."
3,3,Aberdeen_City_231134_DPP,231134_DPP-Neutral-Mr_Martin_Worth_FULL_-22830...,comment_structure,I have read with interest the summary document...,"[""Consultation, Transparency and Information""]",train,"[Consultation, Transparency and Information]"
4,4,Aberdeen_City_231134_DPP,231134_DPP-Neutral-Mr_Sandy_Milne_FULL_-228488...,comment_structure,I note from a review of the Planning Statement...,"[""Grid Connection and Electrical Infrastructure""]",validation,[Grid Connection and Electrical Infrastructure]


In [ ]:
mlb = joblib.load(mlb_path)
label_list = list(mlb.classes_)

print(f"Number of labels: {len(label_list)}")
label_list


Number of labels: 20


['Agricultural Land',
 'Community and Economic Benefits',
 'Consultation, Transparency and Information',
 'Cumulative Impact',
 'Decommissioning and Site Restoration',
 'Emergency Planning and Response',
 'Fire and Explosion Risk',
 'Grid Connection and Electrical Infrastructure',
 'Health and Wellbeing',
 'Landscape, Visual and Heritage Impact',
 'Light Pollution',
 'Noise',
 'Planning Policy and Regulatory Compliance',
 'Project Need',
 'Property Value',
 'Residential Proximity and Separation Distance',
 'Site Selection',
 'Traffic',
 'Water and Soil Contamination',
 'Wildlife and Ecology']

In [ ]:
scotbess_df_train = scotbess_df[scotbess_df["split"] == "train"].copy()
scotbess_df_val = scotbess_df[scotbess_df["split"] == "validation"].copy()
scotbess_df_test = scotbess_df[scotbess_df["split"] == "test"].copy()

print(f"Train: {len(scotbess_df_train)}")
print(f"Validation: {len(scotbess_df_val)}")
print(f"Test: {len(scotbess_df_test)}")

assert len(scotbess_df_train) == 1340
assert len(scotbess_df_val) == 165
assert len(scotbess_df_test) == 170
assert len(scotbess_df_train) + len(scotbess_df_val) + len(scotbess_df_test) == len(scotbess_df)


Train: 1340
Validation: 165
Test: 170


In [ ]:
avg_labels_per_sample = scotbess_df["label_list"].apply(len).mean()
print("Average labels per sample:", avg_labels_per_sample)

Average labels per sample: 5.987462686567164


In [ ]:
# checking if there is no exact text overlap between the frozen splits
train_texts_set = set(scotbess_df_train["final_masked_text"])
val_texts_set = set(scotbess_df_val["final_masked_text"])
test_texts_set = set(scotbess_df_test["final_masked_text"])

train_val_overlap = train_texts_set & val_texts_set
train_test_overlap = train_texts_set & test_texts_set
val_test_overlap = val_texts_set & test_texts_set

print(f"Train-val overlap: {len(train_val_overlap)} examples")
print(f"Train-test overlap: {len(train_test_overlap)} examples")
print(f"Val-test overlap: {len(val_test_overlap)} examples")


Train-val overlap: 0 examples
Train-test overlap: 0 examples
Val-test overlap: 0 examples


In [ ]:
# reuse the fixed label mapping created during Scot-BESS splitting
joblib.dump(mlb, f"{output_dir}/mlb.joblib", compress=3)


['/content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_SVC_final/mlb.joblib']

In [ ]:
scotbess_y_train = mlb.transform(scotbess_df_train["label_list"])
scotbess_y_val = mlb.transform(scotbess_df_val["label_list"])
scotbess_y_test = mlb.transform(scotbess_df_test["label_list"])

print(scotbess_y_train[:1])


[[0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 1 1 0 0 0]]


In [ ]:
scotbess_y_train.shape, scotbess_y_val.shape, scotbess_y_test.shape


((1340, 20), (165, 20), (170, 20))

In [ ]:
label_counts = scotbess_y_train.sum(axis=0)
min_count = label_counts.min()
max_count = label_counts.max()

print("Minimum training examples for a label:", min_count)
print("Maximum training examples for a label:", max_count)


Minimum training examples for a label: 118
Maximum training examples for a label: 728


In [ ]:
print(label_counts)
print(f"Average number of training samples per label: {label_counts.mean():.2f}")


[249 235 363 381 152 412 728 139 509 702 215 528 362 258 118 488 670 559
 387 552]
Average number of training samples per label: 400.35


In [ ]:
scotbess_X_train = scotbess_df_train["final_masked_text"].astype(str)
scotbess_X_val = scotbess_df_val["final_masked_text"].astype(str)
scotbess_X_test = scotbess_df_test["final_masked_text"].astype(str)

scotbess_X_train.shape, scotbess_X_val.shape, scotbess_X_test.shape


((1340,), (165,), (170,))

In [ ]:
# checking the length of consultation responses in the dataset
df_len = pd.concat([
    pd.DataFrame({"text": scotbess_X_train, "split": "train"}),
    pd.DataFrame({"text": scotbess_X_val, "split": "validation"}),
    pd.DataFrame({"text": scotbess_X_test, "split": "test"})], ignore_index=True)

df_len["word_len"] = df_len["text"].str.split().str.len()
df_len[["word_len"]].describe()


,word_len
count,1675.000000
mean,447.967761
std,760.190924
min,1.000000
25%,63.000000
50%,164.000000
75%,465.000000
max,6591.000000


In [ ]:
# inspecting the shortest responses
df_len.nsmallest(10, "word_len")[["split", "word_len", "text"]]


,split,word_len,text
8,train,1,Object
205,train,2,[PLACE] landscape
39,train,3,Unknown health risks.
724,train,3,I object fully
174,train,4,Too close to housing
259,train,4,Will be an eyesore!
1367,validation,4,Flawed and unwanted application.
72,train,5,Proximity to [PLACE] fields area.
190,train,5,"Environmental, noise pollution, traffic conges..."
546,train,5,intrusive and concerns of safety


# SVC baseline


In [ ]:
print("CPU cores:", os.cpu_count())


CPU cores: 2


In [ ]:
svc_model = Pipeline([("tfidf", TfidfVectorizer(analyzer="word", lowercase=True)),
                    ("clf", OneVsRestClassifier(LinearSVC(dual="auto", max_iter=2000, random_state=42)))])

params = {"tfidf__max_features": [10000, 20000], #lower than AAPD because Scot-BESS is much smaller
          "tfidf__ngram_range": [(1,1), (1,2)],
          "tfidf__min_df": [1, 2],
          "tfidf__max_df": [0.95],
          "clf__estimator__class_weight": ["balanced"],
          "clf__estimator__C": [0.1, 1, 10]}

gs = GridSearchCV(
    svc_model,
    params,
    cv=3,
    n_jobs=2,
    verbose=2,
    refit=True,
    scoring="f1_macro")

start_train = time.perf_counter()

gs.fit(scotbess_X_train, scotbess_y_train)

end_train = time.perf_counter()
train_time = end_train - start_train
print(f"Training time: {train_time:.2f} seconds.")

joblib.dump(gs.best_estimator_, f"{output_dir}/SCOTBESS_SVC_gs_best_model.joblib", compress=5)

print(gs.best_params_)
print(gs.best_score_)


Fitting 3 folds for each of 24 candidates, totalling 72 fits
Training time: 153.37 seconds.
{'clf__estimator__C': 1, 'clf__estimator__class_weight': 'balanced', 'tfidf__max_df': 0.95, 'tfidf__max_features': 10000, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 1)}
0.700606368020143


In [ ]:
gs.best_estimator_


Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.95, max_features=10000, min_df=2)),
                ('clf',
                 OneVsRestClassifier(estimator=LinearSVC(C=1,
                                                         class_weight='balanced',
                                                         max_iter=2000,
                                                         random_state=42)))])

In [ ]:
# unfitted copy of the best pipeline found by grid search
final_model = clone(gs.best_estimator_)
final_train_start = time.perf_counter()
final_model.fit(scotbess_X_train, scotbess_y_train)
final_train_time_sec = time.perf_counter() - final_train_start

joblib.dump(final_model, f"{output_dir}/SCOTBESS_SVC_final_model.joblib", compress=5)

print(f"Final best-configuration training time: {final_train_time_sec:.2f} seconds")


Final best-configuration training time: 1.26 seconds


In [ ]:
scotbess_y_val_pred = final_model.predict(scotbess_X_val)

val_miF1 = f1_score(scotbess_y_val, scotbess_y_val_pred, average="micro")
val_maF1 = f1_score(scotbess_y_val, scotbess_y_val_pred, average="macro")

print("Validation Micro-F1:", val_miF1)
print("Validation Macro-F1:", val_maF1)


Validation Micro-F1: 0.8186785891703925
Validation Macro-F1: 0.7867722764794988


In [ ]:
start_inf = time.perf_counter()
scotbess_y_test_pred = final_model.predict(scotbess_X_test)
end_inf = time.perf_counter()

inference_time = end_inf - start_inf
print(f"Inference time: {inference_time:.3f} seconds")

test_miF1 = f1_score(scotbess_y_test, scotbess_y_test_pred, average="micro")
test_maF1 = f1_score(scotbess_y_test, scotbess_y_test_pred, average="macro")

print("Test Micro-F1:", test_miF1)
print("Test Macro-F1:", test_maF1)


Inference time: 0.079 seconds
Test Micro-F1: 0.8149606299212598
Test Macro-F1: 0.782547012229492


In [ ]:
per_sample_ms = (inference_time / len(scotbess_X_test)) * 1000
print(f"Per-sample inference: {per_sample_ms:.4f} ms")


Per-sample inference: 0.4657 ms


In [ ]:
print(classification_report(
    scotbess_y_test,
    scotbess_y_test_pred,
    target_names=mlb.classes_,
    zero_division=0))


                                               precision    recall  f1-score   support

                            Agricultural Land       0.67      0.65      0.66        31
              Community and Economic Benefits       0.86      0.83      0.84        29
   Consultation, Transparency and Information       0.73      0.73      0.73        45
                            Cumulative Impact       0.71      0.85      0.78        47
         Decommissioning and Site Restoration       0.62      0.84      0.71        19
              Emergency Planning and Response       0.82      0.90      0.86        51
                      Fire and Explosion Risk       0.96      0.87      0.91        91
Grid Connection and Electrical Infrastructure       0.56      0.59      0.57        17
                         Health and Wellbeing       0.79      0.78      0.78        63
        Landscape, Visual and Heritage Impact       0.87      0.84      0.85        91
                              Light Pollut

In [ ]:
report_dict = classification_report(
    scotbess_y_test,
    scotbess_y_test_pred,
    target_names=mlb.classes_,
    zero_division=0,
    output_dict=True)

report_df = pd.DataFrame(report_dict).T
report_df.to_csv(f"{output_dir}/SCOTBESS_SVC_classification_report.csv")


In [ ]:
result = {
    "model": "LinearSVC_TFIDF",
    "dataset": "Scot-BESS",
    "best_params": json.dumps(gs.best_params_),
    "cv_score": gs.best_score_,
    "gs_refit_time_sec": train_time, #grid search + refit on train
    "final_train_time_sec": final_train_time_sec, #one fit of the selected config
    "val_f1_micro": val_miF1,
    "val_f1_macro": val_maF1,
    "test_f1_micro": test_miF1,
    "test_f1_macro": test_maF1,
    "inference_time_sec": inference_time, #test set
    "inference_per_sample_ms": per_sample_ms}

results_df = pd.DataFrame([result])
results_df.to_csv(f"{output_dir}/SCOTBESS_SVC_results.csv", index=False)
